In [24]:
import pandas as pd
import os

In [7]:
df_leitos = pd.read_csv(r'C:\Arquivos Vsco\Projeto\Challenge Orale\data\csv final\df_leitos.csv')

In [13]:
def carregar_populacao_ibge(caminho, ano):
    # skiprows=3 pula as linhas de título/metadado antes da tabela real
    df = pd.read_csv(
        caminho,
        encoding="latin1",
        sep=";",
        skiprows=3,
        thousands=None
    )
    # Colunas vêm como: "Unidade da Federação", "<ano>", "Total" -> renomeia pra padronizar
    df.columns = ["UF_COD_NOME", "POPULACAO", "TOTAL"]
    df = df.drop(columns=["TOTAL"])  # redundante, é igual à coluna do ano

    # Separa "11 Rondônia" em código IBGE da UF (2 dígitos) e nome
    df["CO_UF_IBGE"] = df["UF_COD_NOME"].str.extract(r"^(\d+)")
    df["NOME_UF"] = df["UF_COD_NOME"].str.replace(r"^\d+\s*", "", regex=True)
    df = df.drop(columns=["UF_COD_NOME"])

    df["ANO_REF"] = ano
    return df

df_pop_2023 = carregar_populacao_ibge(r"C:\Arquivos Vsco\Projeto\Challenge Orale\data\csv a ser tratado\ibge_cnv_projpop2024uf204550187_34_144_173.csv", 2023)
df_pop_2024 = carregar_populacao_ibge(r"C:\Arquivos Vsco\Projeto\Challenge Orale\data\csv a ser tratado\ibge_cnv_projpop2024uf204524187_34_144_173.csv", 2024)
df_pop_2025 = carregar_populacao_ibge(r"C:\Arquivos Vsco\Projeto\Challenge Orale\data\csv a ser tratado\ibge_cnv_projpop2024uf204544187_34_144_173.csv", 2025)

df_populacao = pd.concat([df_pop_2023, df_pop_2024, df_pop_2025], ignore_index=True)
# Remove linhas de rodapé: ficam sem código de UF válido (NaN em CO_UF_IBGE)
df_populacao = df_populacao.dropna(subset=["CO_UF_IBGE"])

print(df_populacao.head(30))
print(df_populacao.shape)

     POPULACAO CO_UF_IBGE              NOME_UF  ANO_REF
0    1740255.0         11             Rondônia     2023
1     876582.0         12                 Acre     2023
2    4240571.0         13             Amazonas     2023
3     695270.0         14              Roraima     2023
4    8616120.0         15                 Pará     2023
5     799124.0         16                Amapá     2023
6    1567191.0         17            Tocantins     2023
7    7003234.0         21             Maranhão     2023
8    3365881.0         22                Piauí     2023
9    9196672.0         23                Ceará     2023
10   3436278.0         24  Rio Grande do Norte     2023
11   4124468.0         25              Paraíba     2023
12   9514483.0         26           Pernambuco     2023
13   3218607.0         27              Alagoas     2023
14   2281994.0         28              Sergipe     2023
15  14828806.0         29                Bahia     2023
16  21247401.0         31         Minas Gerais  

In [18]:
df_leitos["COMP"] = pd.to_datetime(
    df_leitos["COMP"],
    errors="coerce"
)

In [19]:
print(df_leitos["COMP"].dtype)
print(df_leitos["COMP"].head())

datetime64[ns]
0   1970-01-01 00:00:00.000202301
1   1970-01-01 00:00:00.000202301
2   1970-01-01 00:00:00.000202301
3   1970-01-01 00:00:00.000202301
4   1970-01-01 00:00:00.000202301
Name: COMP, dtype: datetime64[ns]


In [22]:
sigla_para_nome = {
    "RO": "Rondônia", "AC": "Acre", "AM": "Amazonas", "RR": "Roraima", "PA": "Pará",
    "AP": "Amapá", "TO": "Tocantins", "MA": "Maranhão", "PI": "Piauí", "CE": "Ceará",
    "RN": "Rio Grande do Norte", "PB": "Paraíba", "PE": "Pernambuco", "AL": "Alagoas",
    "SE": "Sergipe", "BA": "Bahia", "MG": "Minas Gerais", "ES": "Espírito Santo",
    "RJ": "Rio de Janeiro", "SP": "São Paulo", "PR": "Paraná", "SC": "Santa Catarina",
    "RS": "Rio Grande do Sul", "MS": "Mato Grosso do Sul", "MT": "Mato Grosso",
    "GO": "Goiás", "DF": "Distrito Federal"
}


df_leitos["NOME_UF"] = df_leitos["UF"].map(sigla_para_nome)

# Pega a data mais recente disponível dentro de cada ano (mais robusto que comparar só o mês)
data_max_por_ano = df_leitos.groupby("ANO_REF")["COMP"].transform("max")
df_leitos_snapshot = df_leitos[df_leitos["COMP"] == data_max_por_ano]

# Diagnóstico -- roda e me manda esses 3 prints antes de seguir
print("Linhas antes do filtro:", df_leitos.shape[0])
print("Linhas depois do filtro:", df_leitos_snapshot.shape[0])
print("Duplicatas CNES+ANO_REF no snapshot (deve ser 0):", df_leitos_snapshot.duplicated(subset=["CNES", "ANO_REF"]).sum())

# Join final: leitos agregados por UF + população da UF, mesmo ano
df_uf = leitos_por_uf.merge(df_populacao, on=["NOME_UF", "ANO_REF"], how="left")

# Taxa de leitos por 10 mil habitantes -> métrica clássica de capacidade hospitalar
df_uf["LEITOS_POR_10K_HAB"] = (df_uf["TOTAL_LEITOS_EXISTENTES"] / df_uf["POPULACAO"]) * 10000

print(df_uf.sort_values("LEITOS_POR_10K_HAB"))

Linhas antes do filtro: 255843
Linhas depois do filtro: 21397
Duplicatas CNES+ANO_REF no snapshot (deve ser 0): 0
             NOME_UF  ANO_REF  TOTAL_LEITOS_EXISTENTES  TOTAL_LEITOS_SUS  \
72           Sergipe     2023                    45130             32546   
73           Sergipe     2024                    45572             33834   
9           Amazonas     2023                    87032             72760   
10          Amazonas     2024                    88683             72880   
11          Amazonas     2025                    90718             74190   
..               ...      ...                      ...               ...   
64          Rondônia     2024                    65744             49490   
65          Rondônia     2025                    68805             53669   
18  Distrito Federal     2023                   119377             61227   
19  Distrito Federal     2024                   123228             64404   
20  Distrito Federal     2025                   12

In [23]:
# Agrega Leitos por UF + ANO_REF usando o SNAPSHOT (não o df_leitos completo)
leitos_por_uf = df_leitos_snapshot.groupby(["NOME_UF", "ANO_REF"]).agg(
    TOTAL_LEITOS_EXISTENTES=("LEITOS_EXISTENTES", "sum"),
    TOTAL_LEITOS_SUS=("LEITOS_SUS", "sum"),
    QTD_ESTABELECIMENTOS=("CNES", "nunique")
).reset_index()

# Join final: leitos agregados por UF + população da UF, mesmo ano
df_uf = leitos_por_uf.merge(df_populacao, on=["NOME_UF", "ANO_REF"], how="left")

# Taxa de leitos por 10 mil habitantes
df_uf["LEITOS_POR_10K_HAB"] = (df_uf["TOTAL_LEITOS_EXISTENTES"] / df_uf["POPULACAO"]) * 10000

print(df_uf.sort_values("LEITOS_POR_10K_HAB")[["NOME_UF", "ANO_REF", "TOTAL_LEITOS_EXISTENTES", "POPULACAO", "LEITOS_POR_10K_HAB"]])

             NOME_UF  ANO_REF  TOTAL_LEITOS_EXISTENTES  POPULACAO  \
72           Sergipe     2023                     3766  2281994.0   
73           Sergipe     2024                     3841  2291077.0   
9           Amazonas     2023                     7387  4240571.0   
11          Amazonas     2025                     7634  4321616.0   
10          Amazonas     2024                     7569  4281209.0   
..               ...      ...                      ...        ...   
64          Rondônia     2024                     5500  1746227.0   
65          Rondônia     2025                     5936  1751950.0   
18  Distrito Federal     2023                    10108  2967543.0   
20  Distrito Federal     2025                    10531  2996899.0   
19  Distrito Federal     2024                    10686  2982818.0   

    LEITOS_POR_10K_HAB  
72           16.503111  
73           16.765041  
9            17.419824  
11           17.664688  
10           17.679585  
..                 ..

In [25]:
os.makedirs("data/processed", exist_ok=True)

# Base de população limpa (reutilizável)
df_populacao.to_csv(
    "data/processed/ibge_populacao_uf_2023_2025.csv",
    index=False,
    encoding="utf-8"
)

# Dataset unificado: leitos agregados por UF + população + taxa per capita
df_uf.to_csv(
    "data/processed/leitos_populacao_uf_2023_2025.csv",
    index=False,
    encoding="utf-8"
)

print("População:", df_populacao.shape)
print("Leitos x População (UF):", df_uf.shape)

População: (81, 4)
Leitos x População (UF): (81, 8)
